# 📖 Notebook 3: Building a Snowflake ID Generator

**Snowflake IDs** are a 64-bit flavour of time-ordered ID invented at Twitter. Discord, Instagram, and many others use variants.

Compared to UUID/ULID/KSUID, Snowflake is:

- **Much smaller** (64 bits vs 128/160)
- **Always a plain integer** (no encoding gymnastics, fits in `BIGINT` columns)
- **Strictly time-ordered per worker**, with no randomness

The trade-off: it relies on **wall-clock time** and **unique worker IDs**, so it needs a tiny bit of operational discipline.

## Learning Objectives

- Understand the 64-bit Snowflake bit layout
- Build a working Snowflake generator with sequence overflow handling
- See **clock-skew** defences (what happens if the clock jumps backwards?)
- Decode an existing Snowflake ID back into its parts


## 🛠️ Setup

This lab uses **only the Python standard library + `pydantic`**. No Docker, no Redis, no Postgres.

```bash
cd 01-foundations/id-generation
uv sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


## 🧬 The Snowflake bit layout

Twitter's original scheme packs 64 bits like this:

```
| 1 |      41 bits       |  10 bits  |  12 bits  |
| 0 |  timestamp (ms)    | worker_id |  sequence |
```

- **1 sign bit** — always 0 so the number is a positive signed 64-bit integer (friendly to Java `long`, Postgres `BIGINT`, etc.)
- **41 bits timestamp** — milliseconds since a **custom epoch** (e.g. your app's launch date). 2⁴¹ ms ≈ **69.7 years**.
- **10 bits worker_id** — identifies the process/server. 2¹⁰ = **1024** workers.
- **12 bits sequence** — counter that resets every millisecond. 2¹² = **4096** IDs per worker per ms → **~4.1 million IDs/sec per worker**.

Why those specific sizes? Twitter chose them for their scale. Nothing stops you from re-slicing for your needs (Instagram uses a 41/13/10 layout, for example).


## 🏗️ Step 1 — the layout constants

Let's translate the diagram above into Python constants, then build generation and decoding around them.


In [ ]:
# Bit widths
TIMESTAMP_BITS = 41
WORKER_BITS    = 10
SEQUENCE_BITS  = 12

# Maximum values per field
MAX_TIMESTAMP  = (1 << TIMESTAMP_BITS) - 1   # 2,199,023,255,551
MAX_WORKER     = (1 << WORKER_BITS) - 1      # 1023
MAX_SEQUENCE   = (1 << SEQUENCE_BITS) - 1    # 4095

# Bit shift offsets
WORKER_SHIFT    = SEQUENCE_BITS              # 12
TIMESTAMP_SHIFT = SEQUENCE_BITS + WORKER_BITS  # 22

# Custom epoch: pick any sensible start date. Using 2024-01-01 UTC here.
#   1704067200000 ms = 2024-01-01T00:00:00Z
CUSTOM_EPOCH_MS = 1704067200000

print(f"Max timestamp:  {MAX_TIMESTAMP:,} ms  (~{MAX_TIMESTAMP/1000/60/60/24/365:.1f} years)")
print(f"Max worker id:  {MAX_WORKER}")
print(f"Max sequence:   {MAX_SEQUENCE}  (per ms, per worker)")

# The layout must fit a *signed* 64-bit integer: 1 sign bit + 63 usable bits.
assert TIMESTAMP_BITS + WORKER_BITS + SEQUENCE_BITS == 63
assert (MAX_TIMESTAMP << TIMESTAMP_SHIFT) | (MAX_WORKER << WORKER_SHIFT) | MAX_SEQUENCE == (1 << 63) - 1

## 🏗️ Step 2 — a minimal generator

The rules:

1. Read current time in ms (since our custom epoch).
2. If it's the **same ms as last call**, increment the sequence. If the sequence overflows (> 4095), **spin-wait** until the next ms.
3. If it's a **new ms**, reset the sequence to 0.
4. Compose the 64-bit integer: `(ts << 22) | (worker << 12) | seq`.


In [ ]:
import time, threading
from pydantic import BaseModel, Field

class SnowflakeConfig(BaseModel):
    worker_id: int = Field(..., ge=0, le=MAX_WORKER)
    epoch_ms: int = CUSTOM_EPOCH_MS

class SnowflakeGenerator:
    def __init__(self, config: SnowflakeConfig):
        self.cfg = config
        self._lock = threading.RLock()  # reentrant so subclasses can wrap next_id()
        self._last_ts = -1
        self._sequence = 0

    def _now_ms(self) -> int:
        return int(time.time() * 1000) - self.cfg.epoch_ms

    def next_id(self) -> int:
        with self._lock:
            ts = self._now_ms()
            if ts == self._last_ts:
                # Same millisecond: bump sequence
                self._sequence = (self._sequence + 1) & MAX_SEQUENCE
                if self._sequence == 0:
                    # Sequence overflowed! Spin until the clock ticks.
                    while ts <= self._last_ts:
                        ts = self._now_ms()
            else:
                self._sequence = 0

            self._last_ts = ts

            return (ts        << TIMESTAMP_SHIFT) \
                 | (self.cfg.worker_id << WORKER_SHIFT) \
                 | self._sequence

gen = SnowflakeGenerator(SnowflakeConfig(worker_id=1))
ids = [gen.next_id() for _ in range(5)]
for i in ids:
    print(i)

assert len(set(ids)) == len(ids), "duplicate snowflake"
assert ids == sorted(ids), "snowflakes from one worker must be strictly increasing"
assert all(0 < i < (1 << 63) for i in ids), "must fit a positive signed 64-bit int"
print("\nSorted == creation order? True (asserted)")


## 🧩 Step 3 — decoding a Snowflake

Because each field lives in a known bit range, we can reverse the process and pull the pieces back out.


In [ ]:
from datetime import datetime, timezone

def decode(snowflake: int, epoch_ms: int = CUSTOM_EPOCH_MS) -> dict:
    sequence  =  snowflake        & MAX_SEQUENCE
    worker_id = (snowflake >> WORKER_SHIFT)    & MAX_WORKER
    ts_offset = (snowflake >> TIMESTAMP_SHIFT) & MAX_TIMESTAMP
    unix_ms   = ts_offset + epoch_ms
    return {
        "timestamp_ms": unix_ms,
        "datetime_utc": datetime.fromtimestamp(unix_ms/1000, tz=timezone.utc).isoformat(),
        "worker_id": worker_id,
        "sequence": sequence,
    }


def encode(ts_offset: int, worker_id: int, sequence: int) -> int:
    """Inverse of decode() — useful for proving the two agree."""
    return (ts_offset << TIMESTAMP_SHIFT) | (worker_id << WORKER_SHIFT) | sequence


sample = gen.next_id()
parts = decode(sample)
print("ID:", sample)
print("Decoded:", parts)

# Round-trip: re-encoding the decoded fields must reproduce the ID bit-for-bit.
rebuilt = encode(parts["timestamp_ms"] - CUSTOM_EPOCH_MS, parts["worker_id"], parts["sequence"])
assert rebuilt == sample, (rebuilt, sample)
assert parts["worker_id"] == 1                      # gen was built with worker_id=1
print("\nRe-encoded:", rebuilt, "→ identical to the original ✔")

# Exhaustive check on the corners of every field.
for ts, w, s in [(0, 0, 0), (MAX_TIMESTAMP, MAX_WORKER, MAX_SEQUENCE), (1, 1023, 4095)]:
    d = decode(encode(ts, w, s))
    assert (d["timestamp_ms"] - CUSTOM_EPOCH_MS, d["worker_id"], d["sequence"]) == (ts, w, s)
print("Round-trip holds at every field boundary ✔")

## 💥 Step 4 — seeing sequence overflow in action

To trigger overflow we'd need to emit >4096 IDs in a single ms. On a modern laptop that's easy — but the built-in lock and `time.time()` calls slow us down.

So let's cheat: freeze the clock to a fixed value and request many IDs. Any IDs beyond the first 4096 will force the generator to spin-wait until our fake clock "advances".


In [ ]:
class FrozenClockSnowflake(SnowflakeGenerator):
    """Snowflake whose clock ticks only when we tell it to."""
    def __init__(self, cfg: SnowflakeConfig):
        super().__init__(cfg)
        self._fake_ms = 1_000_000  # arbitrary

    def _now_ms(self) -> int:
        return self._fake_ms

    def tick(self, n=1):
        self._fake_ms += n

demo = FrozenClockSnowflake(SnowflakeConfig(worker_id=7))

# Generate the full sequence (4096) in one ms — should all succeed.
same_ms_ids = [demo.next_id() for _ in range(MAX_SEQUENCE + 1)]
print(f"Emitted {len(same_ms_ids)} IDs in one ms")

assert len(set(same_ms_ids)) == MAX_SEQUENCE + 1, "IDs within one ms must be unique"
assert [decode(i)["sequence"] for i in same_ms_ids] == list(range(MAX_SEQUENCE + 1))
assert len({decode(i)["timestamp_ms"] for i in same_ms_ids}) == 1, "all in the same ms"
print(f"Sequences ran 0..{decode(same_ms_ids[-1])['sequence']}, all inside one millisecond ✔")

# The 4097th ID in this millisecond cannot exist: the sequence field is full.
# The generator spin-waits for the clock to tick, so schedule a tick to release it.
import threading
threading.Timer(0.05, lambda: demo.tick(1)).start()

next_id = demo.next_id()
print("\nID after overflow:", next_id)
print("Decoded:", decode(next_id))

# The guarantee: it waited for a new millisecond rather than reusing a sequence.
assert decode(next_id)["timestamp_ms"] > decode(same_ms_ids[-1])["timestamp_ms"]
assert decode(next_id)["sequence"] == 0
assert next_id not in set(same_ms_ids)

Notice: after overflow, the **timestamp advanced by 1 ms** and the **sequence reset to 0**. That's the guarantee — we never hand out duplicates.


## ⏰ Step 5 — the clock-skew problem

Snowflake's whole correctness argument rests on **the wall clock only moving forward**.

But real wall clocks do **move backwards** sometimes:

- NTP corrections (your server was drifting; NTP yanks it back)
- Manual time changes (admin fat-fingers `date -s`)
- VM pauses / migrations

Our `SnowflakeGenerator` above has no defence. Look at `next_id()` again: if `ts` is
*less* than `_last_ts`, it falls into the `else` branch, resets the sequence to 0, and
happily rewinds `_last_ts`. It re-issues IDs it has already handed out — silently.

Let's not take that on faith. We'll rewind a fake clock and catch the duplicate.

In [ ]:
class FakeClockSnowflake(SnowflakeGenerator):
    """The unmodified generator, with a clock we control."""
    def __init__(self, cfg):
        super().__init__(cfg)
        self._fake_ms = 1_000_000
    def _now_ms(self):
        return self._fake_ms
    def set_clock(self, ms):
        self._fake_ms = ms


naive = FakeClockSnowflake(SnowflakeConfig(worker_id=3))

before = [naive.next_id() for _ in range(3)]   # at ms 1_000_000 -> sequences 0,1,2
naive.set_clock(999_995)                       # 🕐 NTP yanks the clock back 5 ms
during = [naive.next_id() for _ in range(2)]   # the generator just... carries on
naive.set_clock(1_000_000)                     # the clock recovers
after = [naive.next_id() for _ in range(3)]    # "new" millisecond -> sequence restarts at 0

print("before rewind:", before)
print("during rewind:", during)
print("after  rewind:", after)

collisions = sorted(set(before) & set(after))
assert collisions, "expected the naive generator to re-issue IDs after a clock rewind"
print(f"\n❌ {len(collisions)} IDs were handed out TWICE: {collisions}")
print("   As primary keys these are duplicate-key errors — or worse, silent overwrites.")

In [ ]:
class ClockSafeSnowflake(SnowflakeGenerator):
    """The standard defence: refuse to issue rather than risk a duplicate."""

    def next_id(self) -> int:
        # NOTE: `_lock` is an RLock, so we can hold it across the super() call.
        # Checking the clock and then releasing the lock would be a race: another
        # thread could slip in between the check and the actual generation.
        with self._lock:
            ts = self._now_ms()
            if ts < self._last_ts:
                drift_ms = self._last_ts - ts
                raise RuntimeError(
                    f"Clock moved backwards by {drift_ms} ms! Refusing to generate ID."
                )
            return super().next_id()

class FakeClockSafeSnowflake(ClockSafeSnowflake):
    def __init__(self, cfg):
        super().__init__(cfg)
        self._fake_ms = 1_000_000
    def _now_ms(self):
        return self._fake_ms
    def set_clock(self, ms):
        self._fake_ms = ms

safe = FakeClockSafeSnowflake(SnowflakeConfig(worker_id=3))
issued = [safe.next_id() for _ in range(3)]
print("Normal IDs:", issued)

# Pretend the system clock jumped back by 5 seconds:
safe.set_clock(safe._fake_ms - 5000)
try:
    safe.next_id()
    raise AssertionError("a clock-safe generator must refuse, not silently continue")
except RuntimeError as e:
    print("🚨", e)

# Once the clock catches up, generation resumes — and never repeats an old ID.
safe.set_clock(1_000_001)
resumed = [safe.next_id() for _ in range(3)]
assert not (set(issued) & set(resumed)), "clock-safe generator emitted a duplicate"
print("After the clock caught up:", resumed, "— no overlap with the earlier batch ✔")

### Why not just use `time.monotonic()`?

`time.monotonic()` *never* moves backwards — perfect, right?

Problem: **monotonic time has no absolute meaning**. It counts seconds since the process started (or some arbitrary point). Two Snowflake generators on two different machines would emit IDs from totally different "time" ranges, and the IDs wouldn't sort in true creation order across the cluster.

The trick production systems use:

1. Use the **wall clock** for the timestamp field (so IDs sort across machines).
2. Use **monotonic time** *internally* to detect suspicious wall-clock jumps.
3. If the wall clock moved backwards, refuse-or-wait until it catches up.

This is the same split between "what time is it?" (wall clock) and "how much time has passed?" (monotonic) that you'll see in every distributed system.


## 🎯 Recap

- Snowflake = 1 sign + 41 timestamp + 10 worker + 12 sequence = 64 bits.
- Each worker generates up to **4096 IDs / ms** (~4.1M/sec).
- Overflow → spin-wait for the next ms.
- Clock going backwards → **refuse to generate**. We showed the unguarded generator
  handing out the same IDs twice; refusing is the only safe answer. (Waiting for the
  clock to catch up is the other acceptable option — never "carry on regardless".)

### When to reach for Snowflake

- You want **compact numeric** primary keys (`BIGINT`) rather than strings.
- You operate a **fleet of known workers** (you can assign each a unique `worker_id`).
- You have **NTP-synchronised clocks** and can accept the operational discipline.

### When to prefer UUIDv7 / ULID

- You don't want to manage worker IDs.
- 16 bytes is fine for your storage.
- You want a standardised format with library support in every language.

Both families solve the same underlying problem — **coordination-free, time-ordered IDs** — with different bit budgets and operational trade-offs.
